In [1]:
### PYRAMID SANITY....

import numpy as np
import matplotlib.pyplot as plt
import json
import sys
import math
import h5py
sys.path.append("..")
sys.path.append("../models")
from models.data_handler import data_wrangler
import models.modelbase.mltools as mlt
from modelbase.cascade_late_a import model_cascade2_late_a
from modelbase.cascade_mid import model_cascade2_mid
from modelbase.cascade_early import model_cascade2_early
from modelbase.cascade_early_b import model_cascade2_early_b
from modelbase.vit_pyrencoder import multi_vit
from modelbase.cube_cnn import model_flat2

from osgeo import gdal


In [3]:
loc_prefix = "../models/trained/"
raster_prefix = "../data/aligned_raster/colorado_ml_v10/"
data_locs = ["../data/ml_sets/pyramid_v10/"]
raster_locs = ["ECOSTRESS_WUE_22_4326_0_0.tif", "ECOSTRESS_ESI_22_4326_0_0.tif", "GEDI_AGB_4326_0_0.tif"]

In [16]:
crs = []
y_geolayers = []

### load WUE, ESI, AGB layers
### retain:
###     raster grid size
###     usable-crs, crs, projection
###     nodata value
for i in range(3):
    raster_i = gdal.Open(raster_prefix+raster_locs[i])
    rasterband_i = raster_i.GetRasterBand(1)
    nodata_i = rasterband_i.GetNoDataValue()
    rastersize_i = (raster_i.RasterXSize, raster_i.RasterYSize)
    #tulh, tpxh, _, tulv, _, tpxv
    # tpxv -> abs (tpxv)
    geo_i = raster_i.GetGeoTransform()
    proj_i = raster_i.GetProjection()
    crs.append((geo_i[0], geo_i[3], geo_i[1], abs(geo_i[5])))
    print(geo_i)
    print(proj_i)
    print(" *** ")
    del rasterband_i
    del raster_i

    y_geolayers.append({"nodata":   nodata_i, "rastersize": rastersize_i, 
                        "crs": geo_i, "proj": proj_i})

### Load legal sample index list
lil = np.genfromtxt(data_locs[0] + "/legal_ids.csv", delimiter=",").astype(int)

(-109.05977631862096, 0.0006298418999999995, 0.0, 41.002830640734636, 0.0, -0.0006298418999999991)
GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]
 *** 
(-109.05977631862096, 0.0006298419, 0.0, 41.002830640734636, 0.0, -0.0006298419)
GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]
 *** 
(-109.06, 0.008997741428571428, 0.0, 41.0, 0.0, -0.008997741428571428)
GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM

In [17]:
print(len(lil))
print("legalindex sample:\n", lil[:5])

98809
legalindex sample:
 [[  1 152]
 [  1 153]
 [  1 154]
 [  1 157]
 [  1 158]]


In [18]:
indexedpyramids = []
yoffset = 15
for i in range(3):
    h5temp = h5py.File(data_locs[0] + "/layer_"+str(i+yoffset)+".h5")
    indexedpyramids.append(np.array(h5temp["data"]))
    del h5temp

In [19]:
print("indexedpyramids sample\n", indexedpyramids[0][7].shape)

indexedpyramids sample
 (15, 15)


In [20]:
block_dims = [15, 15, 1]

### convert from coordinates to indices -- from create_pyramid_functions
def geo_idx(cx, cy, geopack): #ulh, ulv, psh, psv):
    ulh, ulv, psh, psv = geopack
    ix = (cx - ulh) / psh
    iy = (ulv - cy) / psv
    return ix, iy

### convert from indices to coordinates -- same...
def idx_geo(ix, iy, geopack): #ulh, ulv, psh, psv):
    ulh, ulv, psh, psv = geopack
    cx = ulh + (ix * psh)
    cy = ulv - (iy * psv)
    return cx, cy

In [ ]:
spatial = [np.zeros((y_geolayers[i]["rastersize"][0], y_geolayers[i]["rastersize"][1])) + y_geolayers[i]["nodata"] for ii in range(3)]
### iterate over all legal samples and retrieve corresponding elts
for j in range(len(lil)):
    for i in range(3):
        ### legal id locations correspond to position in guiding array...
        suli = lil[j][0]
        sulj = lil[j][1]
        if i < 2:
            ### TODO -- is this an accurate inversion??
            ### convert to general crs...
            cx, cy = idx_geo(suli + 0.5, sulj + 0.5, crs[2])
            ### convert back to this layer crs... 
            li, lj = geo_idx(cx, cy, crs[i])
            ### compute the ultimate UL index for this sample
            ### round down to get central pixel UL
            ### ... this is assuming odd dim sizes
            ### subtract dim//2 to get ultimate UL
            suli = int(li) - (block_dims[i] // 2)
            sulj = int(lj) - (block_dims[i] // 2)
            spatial[i][suli:suli+block_dims[i], sulj:sulj+block_dims[i]] = indexedpyramids[i][j]

KeyError: 1

In [ ]:
outname_base = "../visualize/geographic/"
outname_mods = ["_wue", "_esi", "_agb"]
for i in range(3):
    driver_i = gdal.GetDriverByName("GTiff")
    out_i = driver_i.Create(outname_base + "_actual_" + outname_mods[i] + ".tif",
                            spatial[i].shape[0], spatial[i].shape[1], 1, gdal.GDT_Float32)
    out_i.SetGeoTransform(y_geolayers[i][2])
    out_i.SetProjection(y_geolayers[i][3])
    
    out_i.GetRasterBand(1).WriteArray(spatial[i].transpose())
    out_i.GetRasterBand(1).SetNoDataValue(y_geolayers[i][0])

    out_i.FlushCache()
    del out_i
    del driver_i
